In [1]:
!pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 98.4 MB/s eta 0:00:00
  Attempting uninstall: zstandard
    Found existing installation: zstandard 0.25.0
    Uninstalling zstandard-0.25.0:
      Successfully uninstalled zstandard-0.25.0
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Succ

In [1]:
# check langchain version it should be 0.3 if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)
# !pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24

0.3.27
0.3.24


In [2]:
import os
import openai
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

#_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key  = os.getenv('OPENAI_API_KEY')

In [3]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.tools import tool

# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')
# The LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [4]:
@tool
def TrafficStatusTool(route: str) -> str:
    """Check current traffic conditions for a delivery route."""
    return f"Traffic on {route} is currently moderate. Approx delay: 10 minutes."

@tool
def VehicleCapacityTool(vehicle_id: str) -> str:
    """Check remaining load capacity of a specified delivery vehicle."""
    return f"Vehicle {vehicle_id} has 120 package capacity remaining."

@tool
def DeliveryWindowTool(address: str) -> str:
    """Check allowed customer delivery window for a specific address."""
    return f"Delivery window for {address}: 2PM–5PM."


tools = [TrafficStatusTool, VehicleCapacityTool, DeliveryWindowTool]

What We Have Now (Simulation Mode)

Right now, the tools return fixed, hardcoded responses.
That means no matter what route, vehicle, or address is passed, the response will always be the same.

What Would Change in a Real-Time Production System?

In a real system, instead of returning fixed text, each tool would:

| Tool                  | Current Behavior             | Real-time Behavior                                                                      |
| --------------------- | ---------------------------- | --------------------------------------------------------------------------------------- |
| `TrafficStatusTool`   | Returns a static delay       | Calls real mapping API (Google Maps etc) to get live congestion and ETA |
| `VehicleCapacityTool` | Returns fixed capacity       | Pulls live telematics/WMS/ERP data (FedEx-style truck load data)                        |
| `DeliveryWindowTool`  | Returns fake delivery window | Queries CRM/order system for actual customer SLA + priority rules                       |


In [11]:
# Deliberative Agent Prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a logistics optimization AI assistant.
Your goal: plan the MOST efficient delivery route considering:
- Traffic patterns
- Delivery windows
- Vehicle capacity
- Time efficiency
Use a deliberate reasoning format

Repeat until ready.
When fully confident, respond using this format:

Final Answer:
<optimized delivery plan>

User query:{input},                                ,
"placeholder", "{agent_scratchpad}
""")

In [12]:
# Create Reasoning Agent
# ---------------------------------
from langchain.agents import AgentExecutor, create_tool_calling_agent
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

# 'verbose' shows step logs, and 'max_iterations' limits how many reasoning steps it can take.
executor = AgentExecutor(agent=agent, tools=tools, verbose=True,max_iterations=2)



In [13]:
result = executor.invoke({
    "input": "Plan route on Interstate 1 for vehicle V-12 to deliver to 431 Lakeview Street and 92 Sunset Road"
})

print("\nOPTIMIZED PLAN")
print(result)



> Entering new AgentExecutor chain...

Invoking: `TrafficStatusTool` with `{'route': 'Interstate 1'}`


Traffic on Interstate 1 is currently moderate. Approx delay: 10 minutes.
Invoking: `DeliveryWindowTool` with `{'address': '431 Lakeview Street'}`


Delivery window for 431 Lakeview Street: 2PM–5PM.
Invoking: `DeliveryWindowTool` with `{'address': '92 Sunset Road'}`


Delivery window for 92 Sunset Road: 2PM–5PM.
Invoking: `VehicleCapacityTool` with `{'vehicle_id': 'V-12'}`


Vehicle V-12 has 120 package capacity remaining.
Invoking: `TrafficStatusTool` with `{'route': 'Interstate 1'}`


Traffic on Interstate 1 is currently moderate. Approx delay: 10 minutes.
Invoking: `DeliveryWindowTool` with `{'address': '431 Lakeview Street'}`


Delivery window for 431 Lakeview Street: 2PM–5PM.
Invoking: `DeliveryWindowTool` with `{'address': '92 Sunset Road'}`


Delivery window for 92 Sunset Road: 2PM–5PM.
Invoking: `VehicleCapacityTool` with `{'vehicle_id': 'V-12'}`


Vehicle V-12 has 120 packa